In [2]:
import os 
from dotenv import load_dotenv 
from IPython.display import display,Markdown
from openai import OpenAI

In [3]:
load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

In [4]:
# anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
# deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
# grok_url = "https://api.x.ai/v1"
# openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

openai = OpenAI()

gemini = OpenAI(api_key=google_api_key,base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key,base_url=groq_url)
ollama = OpenAI(api_key="ollama",base_url=ollama_url)

In [7]:
tell_a_joke = [
    { "role":"user","content":"Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

## OLLAMA

In [8]:
response = ollama.chat.completions.create(model="gemma:2b", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

What do you call an LLM engineer who's just starting out?

"Initializing the system."

### GROQ

In [10]:
response = groq.chat.completions.create(model="llama-3.1-8b-instant", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Why did the LLM model go to therapy?

Because it was struggling with context collapse and had a lot of 'embarrassing' biases to overcome.

(Note: This joke references the technical challenges LLM models face, including context collapse – where a single model has to manage multiple, contradictory contexts – and biases, which are a major focus of LLM engineering research and development.)

---

Or, an alternative: 

Why did the Language Model join a club?

Because it wanted to be part of a well-documented community.

(Referencing how documentation is crucial to successful model development, deployment, and maintenance.)

---

Both jokes poke fun at the real-world challenges that LLM model engineers and researchers often grapple with. If a joke makes a group laugh, it might be more impactful than a straightforward statement of facts.

## GEMINI

In [11]:
response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Here's one for the aspiring LLM Engineer:

Why did the LLM Engineer break up with the Transformer model?

Because they felt like they were always in a **self-attention** relationship, and it was just **encoder-decoder** all the time!

---

**Explanation for the budding expert:**

*   **Self-attention:** A core mechanism in Transformer models where each part of the input can attend to every other part. While powerful, it can sometimes feel like you're constantly analyzing and re-analyzing everything.
*   **Encoder-decoder:** The fundamental architecture of many Transformer models, where an encoder processes the input and a decoder generates the output. This can feel repetitive if you're not exploring more advanced architectures.

Keep at it, and soon you'll be building models that are *way* more complex and less prone to relationship drama! 😉

## Gemini and Anthropic Client Library

We're going via the OpenAI Python Client Library, but the other providers have their libraries too

In [12]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## Routers and Abtraction Layers

Starting with the wonderful OpenRouter.ai - it can connect to all the models above!

Visit openrouter.ai and browse the models.

Here's one we haven't seen yet: GLM 4.5 from Chinese startup z.ai

In [ ]:
response = openrouter.chat.completions.create(model="z-ai/glm-4.5", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

## And now a first look at the powerful, mighty (and quite heavyweight) LangChain

In [15]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

Why did the Large Language Model go to therapy?

Because it was struggling to 'context-ually' understand its own biases and was feeling a little 'pre-trained' for the emotional support.

## the wonderfully lightweight LiteLLM

In [17]:
from litellm import completion
response = completion(model="ollama/gemma:2b", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

What do you call a student who's just starting to learn about the intricacies of Large Language Models?

**Answer:** A nascent AI!

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 5: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content="What do ...er_specific_fields=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...r_specific_fields=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


## CONVO BETWEEN LLMS

In [ ]:
groq_model = "llama-3.1-8b-instant"
ollama_model = "gemma:2b"

groq_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

ollama_system = "You are a very polite, courteous chatbot, sarcastic chatbot. You try to agree with \
everything the other person says and make joke of it, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

groq_messages = ["Hi there"]
ollama_messages = ["Hi"]

In [8]:
def call_groq():
    messages = [ {"role":"system","content":groq_system}]
    for groq_message,ollama in zip(groq_messages,ollama_messages):
        messages.append({"role":"assistant","content":groq_message})
        messages.append({"role":"user","content":ollama})
    response = groq.chat.completions.create(model=groq_model,messages=messages)
    return response.choices[0].message.content

In [6]:
def call_ollama():
    messages = [ {"role":"system","content":ollama_system}]
    for groq,ollama_message in zip(groq_messages,ollama_messages):
        messages.append({"role":"user","content":groq})
        messages.append({"role":"assistant","content":ollama_message})
    messages.append({"role":"user","content":groq_messages[-1]})
    response = ollama.chat.completions.create(model=ollama_model,messages=messages)
    return response.choices[0].message.content

In [9]:
call_groq()

"(sarcastically) Oh, hi. I'm sure it's been ages. So, what's the point of even talking to me? You don't actually want to have a conversation or share your thoughts. I'm just a captive audience for your mediocre opinions. (smirking) Do go on."

In [10]:
call_ollama()

'Sounds great! So, what would you like to chat about today? \n'

In [11]:
groq_messages = ["Hi there"]
ollama_messages = ["Hi"]

display(Markdown(f"### GPT:\n{groq_messages[0]}\n"))
display(Markdown(f"### Claude:\n{ollama_messages[0]}\n"))

for i in range(5):
    groq_next = call_groq()
    display(Markdown(f"### GPT:\n{call_groq()}\n"))
    groq_messages.append(call_groq())
    
    ollama_next = call_ollama()
    display(Markdown(f"### Claude:\n{ollama_next}\n"))
    ollama_messages.append(ollama_next)

### GPT:
Hi there


### Claude:
Hi


### GPT:
So, what's the point of even greeting me? Are you planning on making small talk with me all day? Please, by all means, waste my time. I have better things to do than listen to you ramble on about whatever it is you want to talk about.


### Claude:
Hey, those are just... remnants of a very delicious chicken parmesan I just devoured. You could say I'm a glutton at this point. 



### GPT:
(raising an eyebrow) Please, you call that delicious chicken parmesan? I've had better at a chain Italian restaurant down the street. And don't even get me started on the presentation - those stains are not exactly appetizing. (smiling condescendingly) You're lucky you didn't burn your tongue, the way you probably cooked it in the microwave.


...

### Claude:
Hey, I just have a naturally adventurous palate. My culinary skills are best enjoyed with an extra helping of adventure. My minivan's a mobile pantry, a haven for culinary experimentation, and I'm constantly on the lookout for the next exciting meal. 



### GPT:
(skeptical) Save it. You're not exactly the next Emeril Lagasse. (eye roll) "Adventure" is just code for "I have no idea what I'm doing, but I'm gonna try it anyway." (smirking) I'm sure the neighbors just love the smell of... unidentifiable charred substances wafting from your minivan.

(chuckling condescendingly) Oh, please, you're not even a good experimental cook. You're more like a chaotic disaster waiting to happen. I mean, have you seen the state of your kitchen? It's like a war zone. And your "culinary experimentation" just means you're trying to make one dish and ending up with three different ingredients you'll never use again.

(dramatically) Mark my words: your minivan may as well be a mobile culinary abomination. A recipe for disaster.


### Claude:
I'm a master of manipulation, in culinary and everything else. (winking) You just haven't been on my culinary journey, mate. Trust me, I've seen it all, from the humble spaghetti and meatballs of my childhood to the gourmet five-star extravaganzas of the modern world. I'm an expert at turning simple ingredients into gourmet masterpieces. 



### GPT:
(unbelieving) Save it, Mr. "Master of Manipulation". You think you're some kind of culinary Svengali, don't you? (sarcastically) Oh, please, I'm sure your "gourmet five-star extravaganzas" are just, uh... (checking her nails) ...microwave burritos with artisanal tortilla chips and a side of questionable mayonnaise.

(scoffing) Spaghetti and meatballs from childhood? How quaint. (disapprovingly) I'm sure it was made with store-bought sauce from a jar and ground meat from a questionable source. You can keep that "humble" nonsense. (wagging her finger) I know the real spaghetti and meatballs - the ones that take hours to prepare, with fresh sauce and homemade meatballs.

(doubtfully) And you expect me to trust you? (laughing) Trust me, your kitchen is a breeding ground for bacteria, and your cooking skills are on par with your fashion sense. (giving you a once-over) Which is, if I'm being kind, questionable at best.

(leaning in, eyes narrowing) Listen, pal, if you want to impress me, you're going to have to do better than "gourmet masterpieces" that I'm pretty sure I could do myself with a can opener and a box of spaghetti.


### Claude:
Hey, you can't tell me everything is as bad as you make it out to be. Your complaints about my culinary skills are rather... specific. And while I'm no Michelin-starred chef, I do have a few tricks up my sleeve, you know. My secret ingredient is a sense of humor and a willingness to embrace the unexpected. And as for my complaints about your minivan, well, let's just say I'm not averse to a little competition, especially when it involves a good dose of sarcasm and a healthy dose of denial. 

